![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [14]:
# QuantConnect Notebook Setup for CSIVMR Options Strategy

from datetime import timedelta
import pandas as pd
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Adjust to terminal width
pd.set_option('display.max_rows', None)  # Show all rows (if needed)

# Initialize QuantBook
qb = QuantBook()
qb.SetBenchmark("SPY")
universe_resolution = Resolution.Daily
start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2024-12-31")

# Define a broad equity universe
tickers = [
    "COUP" , "CSCO", "GILD" , "FHN", "FIS", "T", "BMY", "ATVI", "BAC", "WFC", "F", "MANU", "CMCSA", "ORCL", "SHC",
    # "NEE", "DAL", "DIS", "KO", "AAL", "OXY", "SCHW", "PYPL", "UAL", "MS", "SLB", "GM", "BX", "LVS", "VZ", "OSH", "RTX",
    # "CPRI", "NET", "PLTR", "ALLY", "DVN", "FTNT", "TJX", "C", "TTD", "DASH", "LYFT", "AI", "UBER", "OKTA", "SNAP", "GE",
    # "PFE", "DKNG", "RETA", "ZM", "LCID", "MDT", "U", "IFF", "INTC", "CVS", "RIVN", "GME", "MRVL", "BILL", "PTON", "W",
    # "GOOG", "TRTN", "VFC", "WWE", "MU", "SQ", "NATI", "PM", "DOCU", "CRWD", "COP", "BSX", "KMX", "MMP", "STAG", "PINS",
    # "KVUE", "HZNP", "GEHC", "AMAT", "VKTX", "PDCE", "DDOG", "TSN", "AMZN", "MDLZ", "STT", "USB", "CCK", "DICE", "CPNG",
    # "GDDY", "COIN", "ENB", "DXC", "KR", "ABT", "FAST", "MRTX", "WAL", "CART", "AAP", "IMVT", "CRH", "ED", "PCG", "SRC",
    # "SPY", "TRU", "DELL"
]
all_stock_history = []
all_option_history = []

for stock in tickers:
    symbol = qb.AddEquity(stock, universe_resolution).Symbol
    qb.AddOption(symbol)

    # Attempt to retrieve daily equity data
    eq_data = qb.History([symbol], start_date, end_date, universe_resolution).reset_index()
    if not eq_data.empty:
        eq_data = eq_data[["time","close"]].copy()
        eq_data["symbol"] = stock
        all_stock_history.append(eq_data)

    # Attempt to retrieve daily option data
    opt_data = qb.OptionHistory(symbol, start_date, end_date, universe_resolution).DataFrame.reset_index()
    if not opt_data.empty:
        opt_data["symbol"] = stock
        all_option_history.append(opt_data)

# Suppose df_equity has columns: [time, underlying_price, symbol]
# 1) Concat all equity
df_equity = pd.concat(all_stock_history, ignore_index=True)
# df_equity.rename(columns={"close":"underlying_price"}, inplace=True)

# 2) Concat all options
polished_df_options = pd.concat(all_option_history, ignore_index=True)

# 3) Pivot equity table
# Pivot so that 'time' -> index, 'symbol' -> columns, 'underlying_price' -> values
polished_df_equity = df_equity.pivot(index="time", columns="symbol", values="close")

# Optionally sort by time and drop missing data as you like:
polished_df_equity.sort_index(inplace=True)
polished_df_equity.dropna(how="all", inplace=True)



SAMPLE DATA OBTAINED

In [15]:
exp_polished_df_equity = polished_df_equity[:20]
print(f"polished_df_columns {polished_df_equity.columns}")

exp_polished_df_options = polished_df_options[:20]
print(f"polished_options_columns {polished_df_options.columns}")



ENRICH OPTIONS DATA WITH UNDERLYING PRICES

In [16]:


# 1) Convert it to long format
df_equity_long = (
    polished_df_equity
    .stack()               # pivot columns → row dimension
    .reset_index()         # flatten the multi-index
)
df_equity_long.columns = ["time", "symbol", "underlying_close"]

# 2) Merge it into the options DataFrame on (time, symbol)
enriched_df_options = pd.merge(
    polished_df_options,           # Has columns [time, symbol, expiry, strike, ...]
    df_equity_long,       # Has columns [time, symbol, underlying_close]
    on=["time", "symbol"],
    how="left"            # or "inner"
)

exp_enriched_df_options = enriched_df_options[:20]
print(exp_enriched_df_options)

FUNCTION TO FURTHER ENRICH THE OPTIONS

In [17]:
import math
import numpy as np
import pandas as pd
from scipy.stats import norm

### Black-Scholes helpers
def bs_price(option_type, S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(0, (S - K) if option_type.upper() == 'C' else (K - S))

    d1 = (math.log(S/K) + (r + sigma**2/2)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)

    if option_type.upper() == 'C':
        return S * norm.cdf(d1) - K * math.exp(-r*T)*norm.cdf(d2)
    else:  # Put
        return K * math.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

def bs_implied_vol(option_type, S, K, T, r, option_price, guess=0.2):
    if T <= 0:
        return 0.0
    eps = 1.0e-5
    max_iter = 50
    sigma = guess

    for i in range(max_iter):
        price = bs_price(option_type, S, K, T, r, sigma)
        if price == option_price:
            return sigma
        d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T)/(sigma*math.sqrt(T))
        vega = S*norm.pdf(d1)*math.sqrt(T)
        if vega < 1.0e-8:
            return sigma
        sigma -= (price - option_price)/vega
        if abs(price - option_price) < eps:
            break
        if sigma <= 0:
            sigma = eps
    return abs(sigma)

def compute_option_columns(row, r=0.02):
    opt_type = 'C' if row["type"] == 0 else 'P'
    S = row.get("underlying_close", 0.0)
    K = row.get("strike", 0.0)

    # Convert times, skip if invalid
    time_ts = pd.to_datetime(row.get("time"), errors="coerce")
    expiry_ts = pd.to_datetime(row.get("expiry"), errors="coerce")

    if (not isinstance(time_ts, pd.Timestamp)) or (not isinstance(expiry_ts, pd.Timestamp)):
        return pd.Series({"days_to_expiration":0.0,"theoretical_bs_price":0.0,"implied_vol":0.0})

    if pd.isna(time_ts) or pd.isna(expiry_ts):
        return pd.Series({"days_to_expiration":0.0,"theoretical_bs_price":0.0,"implied_vol":0.0})

    # If you only have daily data, skip .normalize() or handle carefully
    days = (expiry_ts - time_ts).days
    if days < 0: days = 0
    T = days/365.0

    # Option price = mid of bid/ask
    if "bidclose" in row and "askclose" in row and not pd.isna(row["bidclose"]):
        opt_price = 0.5*(row["bidclose"] + row["askclose"])
    else:
        opt_price = row.get("lastclose", 0.0)

    if T<=0 or S<=0 or K<=0 or opt_price<=0:
        return pd.Series({"days_to_expiration":days,"theoretical_bs_price":0.0,"implied_vol":0.0})

    # Compute theoretical price with a guess
    implied_v    = bs_implied_vol(opt_type, S, K, T, r, opt_price, guess=0.5)
    theory_price = bs_price(opt_type, S, K, T, r, implied_v)

    return pd.Series({
        "days_to_expiration": float(days),
        "theoretical_bs_price": theory_price,
        "implied_vol": implied_v
    })


# Then apply:
enriched_df_options[["days_to_expiration",
                     "theoretical_bs_price",
                     "implied_vol"]] = enriched_df_options.apply(compute_option_columns, axis=1)



In [18]:
exp_enriched_df_options = enriched_df_options[:100]

In [19]:
micro_enriched_df_options = enriched_df_options[:20]
print(micro_enriched_df_options.head(20))


In [20]:
# Filter a handful of put rows
puts_sample = enriched_df_options[ (enriched_df_options["type"] == 1) & (enriched_df_options["implied_vol"] < 0.001) ]
print(puts_sample[["time","expiry","strike","underlying_close","bidclose","askclose","close","implied_vol"]].head(20))


In [21]:
# The columns you want to display
cols = [
    "expiry", "strike", "type", "symbol", "time",
    "askclose", "bidclose", "close", "open", "volume",
    "underlying_close", "days_to_expiration",
    "theoretical_bs_price", "implied_vol"
]

# If you'd like to keep the DataFrame's existing index as a column,
# we can `reset_index(drop=False)` or rename it as "idx".
df_readable = enriched_df_options.reset_index(drop=False)

# Now select just those columns, up to the first 100 rows:
df_readable[cols].head(100)

In [22]:
### FOR DEBUGGING PURPOSES ONLY
example_puts_data  = [
    {
        "type": 1,
        "time": "2024-01-02 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.586625,
        "strike": 50.0,
        "bidclose": 0.48,
        "askclose": 0.51,
        "close": 0.50,
    },
    {
        "type": 1,
        "time": "2024-01-03 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.964692,
        "strike": 50.0,
        "bidclose": 0.37,
        "askclose": 0.40,
        "close": 0.35,
    },
    {
        "type": 1,
        "time": "2024-01-04 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.547847,
        "strike": 50.0,
        "bidclose": 0.49,
        "askclose": 0.53,
        "close": 0.49,
    },
    {
        "type": 1,
        "time": "2024-01-05 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.557542,
        "strike": 50.0,
        "bidclose": 0.47,
        "askclose": 0.49,
        "close": 0.48,
    },
    {
        "type": 1,
        "time": "2024-01-08 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.916222,
        "strike": 50.0,
        "bidclose": 0.27,
        "askclose": 0.30,
        "close": 0.29,
    },
    {
        "type": 1,
        "time": "2024-01-09 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.383048,
        "strike": 50.0,
        "bidclose": 0.48,
        "askclose": 0.50,
        "close": 0.52,
    },
    {
        "type": 1,
        "time": "2024-01-10 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.702952,
        "strike": 50.0,
        "bidclose": 0.33,
        "askclose": 0.36,
        "close": 0.35,
    },
    {
        "type": 1,
        "time": "2024-01-11 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.896834,
        "strike": 50.0,
        "bidclose": 0.21,
        "askclose": 0.24,
        "close": 0.23,
    },
    {
        "type": 1,
        "time": "2024-01-12 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.799893,
        "strike": 50.0,
        "bidclose": 0.21,
        "askclose": 0.23,
        "close": 0.22,
    },
    {
        "type": 1,
        "time": "2024-01-16 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 49.042244,
        "strike": 50.0,
        "bidclose": 0.09,
        "askclose": 0.13,
        "close": 0.11,
    },
]

example_calls_data = [
    {
        "type": 0,
        "time": "2024-01-02 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.586625,
        "strike": 50.0,
        "bidclose": 0.74,
        "askclose": 0.81,
        "close": 0.77,
    },
    {
        "type": 0,
        "time": "2024-01-03 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.964692,
        "strike": 50.0,
        "bidclose": 1.05,
        "askclose": 1.06,
        "close": 1.05,
    },
    {
        "type": 0,
        "time": "2024-01-04 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.547847,
        "strike": 50.0,
        "bidclose": 0.69,
        "askclose": 0.72,
        "close": 0.70,
    },
    {
        "type": 0,
        "time": "2024-01-05 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.557542,
        "strike": 50.0,
        "bidclose": 0.65,
        "askclose": 0.69,
        "close": 0.70,
    },
    {
        "type": 0,
        "time": "2024-01-08 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.916222,
        "strike": 50.0,
        "bidclose": 0.81,
        "askclose": 0.84,
        "close": 0.83,
    },
    {
        "type": 0,
        "time": "2024-01-09 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.383048,
        "strike": 50.0,
        "bidclose": 0.83,
        "askclose": 0.49,
        "close": 0.50,
    },
    {
        "type": 0,
        "time": "2024-01-10 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.702952,
        "strike": 50.0,
        "bidclose": 0.66,
        "askclose": 0.70,
        "close": 0.65,
    },
    {
        "type": 0,
        "time": "2024-01-11 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.896834,
        "strike": 50.0,
        "bidclose": 0.71,
        "askclose": 0.73,
        "close": 0.72,
    },
    {
        "type": 0,
        "time": "2024-01-12 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.799893,
        "strike": 50.0,
        "bidclose": 0.58,
        "askclose": 0.61,
        "close": 0.60,
    },
    {
        "type": 0,
        "time": "2024-01-16 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 49.042244,
        "strike": 50.0,
        "bidclose": 0.68,
        "askclose": 0.74,
        "close": 0.61,
    },
    {
        "type": 0,
        "time": "2024-01-17 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 48.761117,
        "strike": 50.0,
        "bidclose": 0.49,
        "askclose": 0.48,
        "close": 0.49,
    },
    {
        "type": 0,
        "time": "2024-01-18 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 49.197349,
        "strike": 50.0,
        "bidclose": 0.68,
        "askclose": 0.82,
        "close": 0.77,
    },
    {
        "type": 0,
        "time": "2024-01-19 16:00:00",
        "expiry": "2024-01-19",
        "underlying_close": 49.701440,
        "strike": 50.0,
        "bidclose": 1.22,
        "askclose": 1.31,
        "close": 1.24,
    },
]



In [23]:
# FOR DEBUGGING PURPOSES ONLY
import math
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq

def bs_price(option_type, S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        # immediate payoff approximation:
        return max(0, (S - K)) if option_type.upper()=='C' else max(0, (K - S))
    d1 = (math.log(S/K) + (r + sigma**2/2)*T)/(sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    if option_type.upper() == 'C':
        return (S*norm.cdf(d1) - K*math.exp(-r*T)*norm.cdf(d2))
    else:
        return (K*math.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1))

def bs_diff(sigma, opt_type, S, K, T, r, market_price):
    """Difference between BS theoretical price and market price"""
    return bs_price(opt_type, S, K, T, r, sigma) - market_price

def bs_implied_vol_brent(opt_type, S, K, T, r, market_price):
    """Bracketed implied vol solver to avoid Newton overshoot"""
    if T<=0 or market_price<=0 or S<=0 or K<=0:
        return 0.0
    # brentq requires the function to be monotonic over the bracket
    # typical bracket is [1e-5, 5] => from near-zero vol up to 500% vol
    try:
        iv = brentq(bs_diff, 1e-5, 5.0, args=(opt_type, S, K, T, r, market_price), maxiter=200)
        return iv
    except ValueError:
        # can't bracket => fallback
        return 0.0

def compute_iv_bs_for_row(row, r=0.02):
    """
    row: a dictionary containing (type, time, expiry, underlying_close, strike, bidclose, askclose)
    Returns a dictionary with keys:
      days_to_expiration, implied_vol, theoretical_bs_price
    Prints debugging info about calls vs puts.
    """
    # Option type
    opt_type = 'C' if row["type"] == 0 else 'P'

    # Underlying price & strike
    S = row.get("underlying_close", 0.0)
    K = row.get("strike", 0.0)

    # Observed price => mid of bid/ask
    bid = row.get("bidclose", 0.0)
    ask = row.get("askclose", 0.0)
    obs_price = (bid+ask)/2.0 if (bid>0 and ask>0) else row.get("close",0.0)

    # Days to expiry
    time_dt   = pd.to_datetime(row["time"],   errors="coerce")
    expiry_dt = pd.to_datetime(row["expiry"], errors="coerce")
    if pd.isna(time_dt) or pd.isna(expiry_dt):
        print(f"Skipping row: invalid time/expiry => {row}")
        return {
            "days_to_expiration": 0,
            "implied_vol": 0.0,
            "bs_theoretical": 0.0
        }
    days = (expiry_dt - time_dt).days
    if days<0: days=0
    T = days/365.0

    # Debug print each row
    print("\n========== Debug row info ==========")
    print(f"Option Type = {opt_type}, Underlying={S:.3f}, Strike={K}, T={T:.4f} yrs, Observed={obs_price:.3f}")
    print(f"Bid={bid}, Ask={ask}, time={time_dt}, expiry={expiry_dt}, days={days}")

    # If not feasible
    if T<=0 or S<=0 or K<=0 or obs_price<=0:
        print(" => skipping, no valid data or zero price")
        return {
            "days_to_expiration": days,
            "implied_vol": 0.0,
            "bs_theoretical": 0.0
        }

    # compute implied vol via brent
    implied_v = bs_implied_vol_brent(opt_type, S, K, T, r, obs_price)
    # then compute theoretical price
    bs_theor  = bs_price(opt_type, S, K, T, r, implied_v)

    print(f" => ImpliedVol={implied_v:.4f}, BS_Theoretical={bs_theor:.3f}")
    return {
        "days_to_expiration": days,
        "implied_vol": implied_v,
        "bs_theoretical": bs_theor
    }


In [24]:
# (A) Puts
put_rows = []
for row in example_puts_data:
    row_out = compute_iv_bs_for_row(row)
    combined = {**row, **row_out}
    put_rows.append(combined)

puts_result = pd.DataFrame(put_rows)
print("\n==== FINAL Puts Result ====")
print(puts_result[["time","expiry","type","underlying_close","strike","bidclose","askclose","close",
                   "days_to_expiration","implied_vol","bs_theoretical"]])



In [25]:
# (B) Calls
call_rows = []
for row in example_calls_data:
    row_out = compute_iv_bs_for_row(row)
    combined = {**row, **row_out}
    call_rows.append(combined)

calls_result = pd.DataFrame(call_rows)
print("\n==== FINAL Calls Result ====")
print(calls_result[["time","expiry","type","underlying_close","strike","bidclose","askclose","close",
                    "days_to_expiration","implied_vol","bs_theoretical"]])

In [26]:
results_list = []
for row_dict in example_puts_data:
    result_dict = compute_example_row(row_dict)
    # Optionally, unify them for a final row
    combined = {**row_dict, **result_dict}  # merges the original fields + new columns
    results_list.append(combined)

# Convert to DataFrame if needed
results_df = pd.DataFrame(results_list)
print("\n=== Final Results DataFrame ===")
print(results_df.head(10))

In [ ]:
# Let's assume your DataFrame is called option_history 
# and it has columns like:
# [ 'underlying_symbol', 'expiry', 'type', 'askclose', 'bidclose', 'implied_vol', ...]

import pandas as pd

# 1) Group by underlying_symbol, expiry, type
grouped = option_history.groupby(["symbol", "expiry", "type"])

# 2a) For 'askclose' we find idx of min and max in each group
idxmin_ask = grouped['askclose'].idxmin()
idxmax_ask = grouped['askclose'].idxmax()

# 2b) For 'bidclose' we find idx of min and max in each group
idxmin_bid = grouped['bidclose'].idxmin()
idxmax_bid = grouped['bidclose'].idxmax()

# 3) Extract those rows from the original DataFrame
df_min_ask = option_history.loc[idxmin_ask, ["askclose","implied_vol"]].rename(
    columns={"askclose":"min_askclose","implied_vol":"min_askclose_iv"}
)
df_max_ask = option_history.loc[idxmax_ask, ["askclose","implied_vol"]].rename(
    columns={"askclose":"max_askclose","implied_vol":"max_askclose_iv"}
)
df_min_bid = option_history.loc[idxmin_bid, ["bidclose","implied_vol"]].rename(
    columns={"bidclose":"min_bidclose","implied_vol":"min_bidclose_iv"}
)
df_max_bid = option_history.loc[idxmax_bid, ["bidclose","implied_vol"]].rename(
    columns={"bidclose":"max_bidclose","implied_vol":"max_bidclose_iv"}
)

# 4) Combine them all into a single DataFrame:
result_df = (pd.concat([df_min_ask, df_max_ask, df_min_bid, df_max_bid], axis=1)
             .reset_index())

# Now result_df has columns:
# [ 'underlying_symbol', 'expiry', 'type',
#   'min_askclose', 'min_askclose_iv',
#   'max_askclose', 'max_askclose_iv',
#   'min_bidclose', 'min_bidclose_iv',
#   'max_bidclose', 'max_bidclose_iv' ]

print(result_df.head())


In [ ]:
import pandas as pd
import numpy as np

# 1) Confirm columns are correct
print("option_history columns:", option_history.columns.tolist())
# e.g. ["symbol","expiry","type","askclose","bidclose","implied_vol",...]

# 2) Filter out rows with missing askclose or bidclose so min/max is well-defined
clean_df = option_history.dropna(subset=["askclose","bidclose"])

# 3) Group by symbol, expiry, type
grouped = clean_df.groupby(["symbol", "expiry", "type"])

# 4a) Retrieve index of min & max for askclose
idxmin_ask = grouped["askclose"].idxmin()
idxmax_ask = grouped["askclose"].idxmax()

# 4b) Retrieve index of min & max for bidclose
idxmin_bid = grouped["bidclose"].idxmin()
idxmax_bid = grouped["bidclose"].idxmax()

# If any group has no valid data, these can contain NaNs or missing index labels
idxmin_ask = idxmin_ask.dropna()
idxmax_ask = idxmax_ask.dropna()
idxmin_bid = idxmin_bid.dropna()
idxmax_bid = idxmax_bid.dropna()

# 5) For each aggregator, pull the relevant rows from the original DF
df_min_ask = (clean_df.loc[idxmin_ask, ["symbol","expiry","type","askclose","implied_vol"]]
                     .rename(columns={"askclose":"min_askclose","implied_vol":"min_askclose_iv"}))

df_max_ask = (clean_df.loc[idxmax_ask, ["symbol","expiry","type","askclose","implied_vol"]]
                     .rename(columns={"askclose":"max_askclose","implied_vol":"max_askclose_iv"}))

df_min_bid = (clean_df.loc[idxmin_bid, ["symbol","expiry","type","bidclose","implied_vol"]]
                     .rename(columns={"bidclose":"min_bidclose","implied_vol":"min_bidclose_iv"}))

df_max_bid = (clean_df.loc[idxmax_bid, ["symbol","expiry","type","bidclose","implied_vol"]]
                     .rename(columns={"bidclose":"max_bidclose","implied_vol":"max_bidclose_iv"}))

# 6) Merge them side by side
#    We'll do a series of merges on [symbol, expiry, type] so we keep those columns.
summary_df = (df_min_ask
  .merge(df_max_ask, on=["symbol","expiry","type"], how="outer")
  .merge(df_min_bid, on=["symbol","expiry","type"], how="outer")
  .merge(df_max_bid, on=["symbol","expiry","type"], how="outer")
)

summary_df.head(20)

In [ ]:
import pandas as pd
import unittest

def transform_option_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform the raw options DataFrame such that for each group of 
    (underlying_symbol, expiry, time):
      - Create a new column 'underlying_symbol' by extracting the part
        of symbol.Value before any first space.
      - Pick the call with the lowest implied_vol (plus its columns).
      - Pick the put with the highest implied_vol (plus its columns).
      - Combine them into a single row, renaming columns suitably.
    """
    # 1) Create a new column with the underlying symbol 
    #    using symbol.Value instead of symbol itself.
    def extract_underlying(sym_obj):
        # sym_obj is a QuantConnect Symbol. We can safely call sym_obj.Value (string).
        val = sym_obj.Value
        return val.split(' ')[0] if ' ' in val else val

    df['underlying_symbol'] = df['symbol'].apply(extract_underlying)

    # Separate calls and puts
    calls_df = df[df['type'] == 0].copy()
    puts_df  = df[df['type'] == 1].copy()
    
    # 2) For calls, pick row with the lowest implied_vol 
    #    within each (underlying_symbol, expiry, time).
    calls_min_iv = calls_df.loc[
        calls_df.groupby(['underlying_symbol', 'expiry', 'time'])['implied_vol'].idxmin()
    ].copy()
    
    # Rename columns to distinguish call data
    call_col_map = {
        'strike': 'call_strike',
        'implied_vol': 'call_implied_vol',
        'bidopen': 'call_bidopen',
        'bidclose': 'call_bidclose',
        'askopen': 'call_askopen',
        'askclose': 'call_askclose',
    }
    calls_min_iv.rename(columns=call_col_map, inplace=True)

    # 3) For puts, pick row with the highest implied_vol
    #    within each (underlying_symbol, expiry, time).
    puts_max_iv = puts_df.loc[
        puts_df.groupby(['underlying_symbol', 'expiry', 'time'])['implied_vol'].idxmax()
    ].copy()
    
    # Rename columns to distinguish put data
    put_col_map = {
        'strike': 'put_strike',
        'implied_vol': 'put_implied_vol',
        'bidopen': 'put_bidopen',
        'bidclose': 'put_bidclose',
        'askopen': 'put_askopen',
        'askclose': 'put_askclose',
    }
    puts_max_iv.rename(columns=put_col_map, inplace=True)

    # 4) Merge calls and puts on (underlying_symbol, expiry, time)
    merged_df = pd.merge(
        calls_min_iv,
        puts_max_iv,
        on=['underlying_symbol', 'expiry', 'time'],
        how='inner'
    )
    print('calls')
    print(calls_min_iv.columns)
    print(calls_min_iv.head(20))
    print('puts')
    print(puts_max_iv.columns)
    print(puts_max_iv.head(20))

    return merged_df

In [ ]:
transformed_options_history = transform_option_data(option_history)
print (transformed_options_history.head())

In [ ]:
from IPython.display import display, HTML

def filter_and_display_options_history(df):
    """
    Filters the DataFrame to only include rows where 'Expiration' >= '2025-01-01',
    then displays it as HTML in the notebook.
    """
    # Adjust filter to your needs
    df['expiry'] = pd.to_datetime(df['expiry'], errors='coerce')
    # Filter by Expiration date
    filtered_df = df[df['expiry'] >= pd.Timestamp('2025-01-01')]
    
    display(HTML(filtered_df.to_html()))

# Example usage:
filter_and_display_options_history(transformed_options_history)


In [ ]:
option_history.to_csv('option_history.csv', index=False)
from IPython.display import FileLink
FileLink('option_history.csv')

In [ ]:
html_table = option_history.to_html()
with open('option_history.html', 'w', encoding='utf-8') as f:
    f.write(html_table)

from IPython.display import FileLink
FileLink('option_history.html')

In [ ]:
import base64
from IPython.display import HTML

def create_download_link(df, title="Download CSV file", filename="option_history.csv"):
    """
    Convert the DataFrame to CSV, base64-encode it, and return an HTML link
    that you can click to download the CSV directly in the browser.
    """
    csv_data = df.to_csv(index=False)
    b64_data = base64.b64encode(csv_data.encode()).decode()
    href = f'<a href="data:text/csv;base64,{b64_data}" download="{filename}">{title}</a>'
    return HTML(href)

# Example usage with your DataFrame:
create_download_link(option_history)


Unit Test

In [ ]:
# ------------------ Unit Test ------------------
class TestTransformOptionData(unittest.TestCase):
    def test_transform_option_data(self):
        """
        Tests the transform_option_data function using a small, synthetic DataFrame.
        We verify that it correctly creates the 'underlying_symbol' column,
        picks the lowest IV call and highest IV put for each grouping, 
        and merges them properly.
        """
        # Create a small test DataFrame
        data = {
            'symbol':       ['AAPL XYZZ','AAPL ABC','AAPL LMNO','MSFT QWER','MSFT ZZZZ','MSFT BBBB'],
            'expiry':       ['2025-01-25','2025-01-25','2025-01-25','2025-01-25','2025-01-25','2025-01-25'],
            'time':         ['2025-01-01','2025-01-01','2025-01-01','2025-01-01','2025-01-01','2025-01-01'],
            'type':         ['C','C','P','C','P','P'],
            'strike':       [100,105,95,210,205,200],
            'implied_vol':  [0.30,0.38,0.69,0.40,0.41,0.50],
            'bidopen':      [1.0,1.1,2.0,1.2,2.1,2.2],
            'askopen':      [1.2,1.3,2.3,1.3,2.4,2.5],
            'bidclose':     [1.1,1.1,2.1,1.2,2.2,2.3],
            'askclose':     [1.2,1.4,2.4,1.3,2.5,2.6],
        }
        df_test = pd.DataFrame(data)

        # Run transformation
        transformed = transform_option_data(df_test)
        
        # We expect 2 rows in the result:
        #   one for underlying_symbol='AAPL', one for underlying_symbol='MSFT'.
        self.assertEqual(len(transformed), 2)

        # Check that the 'underlying_symbol' was extracted properly
        self.assertIn('underlying_symbol', transformed.columns)

        # For AAPL:
        row_aapl = transformed[transformed['underlying_symbol'] == 'AAPL'].iloc[0]
        # The lowest IV call among calls (0.30, 0.38) is 0.30
        self.assertAlmostEqual(row_aapl['call_implied_vol'], 0.30)
        # The highest IV put among puts (0.69) is 0.69
        self.assertAlmostEqual(row_aapl['put_implied_vol'], 0.69)

        # For MSFT:
        row_msft = transformed[transformed['underlying_symbol'] == 'MSFT'].iloc[0]
        # The lowest IV call is 0.40 (only one call)
        self.assertAlmostEqual(row_msft['call_implied_vol'], 0.40)
        # The highest IV put among puts (0.41, 0.50) is 0.50
        self.assertAlmostEqual(row_msft['put_implied_vol'], 0.50)


if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)